# Ayo Olopon: 10,000 random-policy games

This notebook evaluates two agents that choose uniformly from the legal actions available to them. The cells are separated so each requested metric can be inspected independently.


## 1. Imports and configuration

The local random generator makes the evaluation reproducible without changing Python's global random state. Importing the model registers the game with OpenSpiel.

In [1]:
import sys
from pathlib import Path
import json

sys.path.insert(0, str(Path.cwd().parents[1]))

from Model.ayo_olopon import ayo_olopon
from Model.oware import oware

from dataclasses import dataclass
import random
import pyspiel


In [2]:
NUM_GAMES = 10000
PLAYER_0_RANDOM_SEED = 2
PLAYER_1_RANDOM_SEED = 7
GAME_NAME = 'ayo_olopon'
# GAME_EPISODES_PATH = Path.cwd() / 'game_episodes.json'


## 2. Metrics

One object accumulates results across all games. A crash is counted and the batch continues so the final report includes every requested metric.

In [3]:
@dataclass
class EvaluationMetrics:
    games_completed: int = 0
    crashes: int = 0
    invalid_transitions: int = 0
    illegal_actions_selected: int = 0
    seed_conservation_violations: int = 0
    terminal_state_violations: int = 0

    def as_dict(self):
        return {
            'Games completed': self.games_completed,
            'Crashes': self.crashes,
            'Invalid transitions': self.invalid_transitions,
            'Illegal actions selected': self.illegal_actions_selected,
            'Seed conservation violations': self.seed_conservation_violations,
            'Terminal-state violations': self.terminal_state_violations,
        }

## 3. Invariant helpers

Every initial and resulting state must conserve all seeds. Board and captured-seed values must remain non-negative. A valid action must also change the position and switch players, unless it ends the game.

In [4]:
def seed_total_is_conserved(state):
    # Captured seeds plus board seeds must equal the initial total.
    return (
        sum(state.board) + sum(state.captured) == state.total_seeds
        and all(seeds >= 0 for seeds in state.board)
        and all(seeds >= 0 for seeds in state.captured)
    )


def transition_is_valid(before, after, state):
    # A legal action cannot leave the position unchanged.
    before_board, before_captured, before_player = before
    after_board, after_captured, after_player = after
    if (before_board, before_captured) == (after_board, after_captured):
        return False
    if state.is_terminal():
        return after_player == pyspiel.PlayerId.TERMINAL
    return after_player in (0, 1) and after_player != before_player


def serialize_state(state):
    """Convert a game state into a JSON-friendly dictionary."""
    current_player = state.current_player()
    return {
        'board': list(state.board),
        'captured': list(state.captured),
        'current_player': int(current_player) if current_player != pyspiel.PlayerId.TERMINAL else -1,
        'is_terminal': bool(state.is_terminal()),
        'total_seeds': int(state.total_seeds),
    }


def save_game_episodes(game_episodes, file_path):
    """Save every recorded game as JSON for searchable replay."""
    with open(file_path, 'w', encoding='utf-8') as handle:
        json.dump(game_episodes, handle, indent=2)


## 4. Play one random-policy game

At each turn the policy samples from `state.legal_actions()`. This is random legal play, and the explicit membership check records an illegal-action violation if that contract is ever broken.

In [5]:
def play_random_game(game, player_0_rng, player_1_rng, metrics, episode_id=None, player_0_seed=None, player_1_seed=None):
    state = game.new_initial_state()
    invalid_end = False

    episode = {
        'game_id': episode_id,
        'player_0_seed': player_0_seed,
        'player_1_seed': player_1_seed,
        'initial_state': serialize_state(state),
        'history': [serialize_state(state)],
        'moves': [],
    }

    if not seed_total_is_conserved(state):
        metrics.seed_conservation_violations += 1

    while not state.is_terminal():
        if state.current_player() == pyspiel.PlayerId.TERMINAL:
            metrics.terminal_state_violations += 1
            invalid_end = True
            break

        legal_actions = list(state.legal_actions())
        if not legal_actions:
            metrics.terminal_state_violations += 1
            invalid_end = True
            break

        current_player = int(state.current_player())
        if current_player == 0:
            action = player_0_rng.choice(legal_actions)
        else:
            action = player_1_rng.choice(legal_actions)

        if action not in legal_actions:
            metrics.illegal_actions_selected += 1
            invalid_end = True
            break

        before = (tuple(state.board), tuple(state.captured), state.current_player())
        try:
            state.apply_action(action)
        except ayo_olopon.RelayCycleDetected as error:
            episode['cycle_detected'] = True
            episode['cycle_report'] = error.report
            episode['cycle_move'] = {
                'move_index': len(episode['moves']),
                'player': current_player,
                'action': int(action),
                'state_before': {
                    'board': list(before[0]),
                    'captured': list(before[1]),
                    'current_player': int(before[2]),
                    'is_terminal': False,
                    'total_seeds': int(state.total_seeds),
                },
            }
            episode['final_state'] = serialize_state(state)
            episode['completed'] = False
            return False, episode
        after = (tuple(state.board), tuple(state.captured), state.current_player())

        move_record = {
            'player': current_player,
            'action': int(action),
            'state_before': {
                'board': list(before[0]),
                'captured': list(before[1]),
                'current_player': int(before[2]) if before[2] != pyspiel.PlayerId.TERMINAL else -1,
                'is_terminal': False,
                'total_seeds': int(state.total_seeds),
            },
            'state_after': None,
        }

        if not transition_is_valid(before, after, state):
            metrics.invalid_transitions += 1
        if not seed_total_is_conserved(state):
            metrics.seed_conservation_violations += 1
        if state.is_terminal() and state.current_player() != pyspiel.PlayerId.TERMINAL:
            metrics.terminal_state_violations += 1

        move_record['state_after'] = serialize_state(state)
        episode['moves'].append(move_record)
        episode['history'].append(serialize_state(state))

    episode['final_state'] = serialize_state(state)
    if state.is_terminal():
        episode['completed'] = True
        return True, episode
    if not invalid_end:
        metrics.terminal_state_violations += 1
    episode['completed'] = False
    return False, episode


def build_visualizable_cycle_report(episode):
    """Keep only the cycle while preserving the visualizer episode format."""
    cycle_move = episode['cycle_move']
    report = episode['cycle_report']
    repeated_state = {
        'board': list(report['board']),
        'captured': list(report['captured']),
        'current_player': report['player'],
        'is_terminal': False,
        'total_seeds': cycle_move['state_before']['total_seeds'],
        'cycle': True,
        'description': (
            f"Cycle repeats after {report['cycle_length']} relays"
        ),
    }
    return {
        'game_id': episode.get('game_id'),
        'cycle_detected': True,
        'cycle_move_index': cycle_move['move_index'],
        'initial_state': cycle_move['state_before'],
        'history': [cycle_move['state_before'], repeated_state],
        'moves': [{
            'player': cycle_move['player'],
            'action': cycle_move['action'],
            'state_before': cycle_move['state_before'],
            'state_after': repeated_state,
            'cycle': True,
        }],
        'final_state': repeated_state,
        'cycle_report': report,
    }


def save_relay_cycle_reports(cycle_reports, file_path='relay_cycle_reports.json'):
    """Persist compact, visualizable reports for all detected relay cycles."""
    with open(file_path, 'w', encoding='utf-8') as handle:
        json.dump(cycle_reports, handle, indent=2)
    return file_path


## 5. Run the 10,000-game evaluation

Run this cell to measure the complete batch. It is intentionally separate from all definitions above.

In [6]:
game = pyspiel.load_game(GAME_NAME)
metrics = EvaluationMetrics()
all_game_episodes = []
cycle_reports = []


def save_completed_episode_snapshot(game_episodes, completed_count):
    """Save a snapshot only after a target number of completed games is reached."""
    snapshot_dir = Path.cwd() / 'completed_game_episodes'
    snapshot_dir.mkdir(exist_ok=True)

    existing_snapshots = sorted(snapshot_dir.glob('completed_game_episodes_*.json'))
    next_index = len(existing_snapshots)
    snapshot_path = snapshot_dir / f'completed_game_episodes_{next_index}.json'

    if snapshot_path.exists():
        next_index += 1
        snapshot_path = snapshot_dir / f'completed_game_episodes_{next_index}.json'

    with open(snapshot_path, 'w', encoding='utf-8') as handle:
        json.dump(game_episodes, handle, indent=2)

    print(f"Completed episode snapshot saved to: {snapshot_path} (completed games: {completed_count})")
    return snapshot_path


for game_index in range(NUM_GAMES):
    player_0_rng = random.Random(PLAYER_0_RANDOM_SEED*4 + game_index)
    player_1_rng = random.Random(PLAYER_1_RANDOM_SEED + game_index)

    try:
        completed, episode = play_random_game(
            game,
            player_0_rng,
            player_1_rng,
            metrics,
            episode_id=game_index,
            player_0_seed=PLAYER_0_RANDOM_SEED + game_index,
            player_1_seed=PLAYER_1_RANDOM_SEED + game_index,
        )
        all_game_episodes.append(episode)

        if episode.get('cycle_detected'):
            cycle_reports.append(build_visualizable_cycle_report(episode))
            save_relay_cycle_reports(cycle_reports)
            print(f"Relay cycle report saved for game {episode.get('game_id')}")
            continue

        if completed:
            metrics.games_completed += 1
            print(f"Games completed: {metrics.games_completed}")

            if metrics.games_completed == NUM_GAMES:
                save_completed_episode_snapshot(all_game_episodes, metrics.games_completed)
    except Exception:
        metrics.crashes += 1
        print(f"Crash occurred. Total crashes: {metrics.crashes}")


Games completed: 1
Games completed: 2
Games completed: 3
Games completed: 4
Games completed: 5
Games completed: 6
Games completed: 7
Games completed: 8
Games completed: 9
Games completed: 10
Games completed: 11
Games completed: 12
Games completed: 13
Games completed: 14
Games completed: 15
Games completed: 16
Games completed: 17
Games completed: 18
Games completed: 19
Games completed: 20
Games completed: 21
Games completed: 22
Games completed: 23
Games completed: 24
Games completed: 25
Games completed: 26
Games completed: 27
Games completed: 28
Games completed: 29
Games completed: 30
Games completed: 31
Games completed: 32
Games completed: 33
Games completed: 34
Games completed: 35
Games completed: 36
Games completed: 37
Games completed: 38
Games completed: 39
Games completed: 40
Games completed: 41
Games completed: 42
Games completed: 43
Games completed: 44
Games completed: 45
Games completed: 46
Games completed: 47
Games completed: 48
Games completed: 49
Games completed: 50
Games com

In [7]:

with open('relay_cycle_reports.json', 'r', encoding='utf-8') as handle:
    relay_cycles = json.load(handle)


for first_index in range(len(relay_cycles)):
    for second_index in range(first_index + 1, len(relay_cycles)):
        first_episode = relay_cycles[first_index]
        second_episode = relay_cycles[second_index]

        matches = []

        for first_move_index, first_move in enumerate(first_episode['moves']):
            first_board = tuple(first_move['state_before']['board'])
            first_action = first_move['action']

            for second_move_index, second_move in enumerate(second_episode['moves']):
                second_board = tuple(second_move['state_before']['board'])
                second_action = second_move['action']

                if first_board == second_board and first_action == second_action:
                    matches.append((first_move_index, second_move_index))

        if matches:
            print(
                f"Cycle episodes {first_episode['game_id']} and "
                f"{second_episode['game_id']} share board/action pairs:"
            )
            for first_move_index, second_move_index in matches:
                print(
                    f"  Episode moves {first_move_index} and "
                    f"{second_move_index}"
                )
        else:
            print(
                f"Cycle episodes {first_episode['game_id']} and "
                f"{second_episode['game_id']} have no matching "
                "board/action pairs."
            )

Cycle episodes 4783 and 9511 have no matching board/action pairs.


In [8]:
import importlib
from Model.ayo_olopon import visualizer
importlib.reload(visualizer)
visualizer.visualize_episode(
    relay_cycles[0],
    autoplay=True,interval_ms=2000
)

## 6. Report and enforce the requirements

The quality gate requires exactly 10,000 completed games and zero violations in every other category.

In [9]:
results = metrics.as_dict()
for metric, value in results.items():
    print(f'{metric}: {value:,}')

expected = {
    'Games completed': NUM_GAMES,
    'Crashes': 0,
    'Invalid transitions': 0,
    'Illegal actions selected': 0,
    'Seed conservation violations': 0,
    'Terminal-state violations': 0,
}
assert results == expected, f'Evaluation failed: {results}'


def find_episode_by_game_id(game_episodes, target_game_id):
    for episode in game_episodes:
        if episode.get('game_id') == target_game_id:
            return episode
    return None

example_episode = find_episode_by_game_id(all_game_episodes, 0)
if example_episode is not None:
    print('Example game search succeeded:', example_episode['game_id'])
    print('First move:', example_episode['moves'][0] if example_episode['moves'] else 'No moves recorded')


Games completed: 10,000
Crashes: 0
Invalid transitions: 0
Illegal actions selected: 0
Seed conservation violations: 0
Terminal-state violations: 0
Example game search succeeded: 0
First move: {'player': 0, 'action': 1, 'state_before': {'board': [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], 'captured': [0, 0], 'current_player': 0, 'is_terminal': False, 'total_seeds': 48}, 'state_after': {'board': [6, 2, 7, 1, 6, 1, 6, 6, 6, 0, 1, 6], 'captured': [0, 0], 'current_player': 1, 'is_terminal': False, 'total_seeds': 48}}
